In [2]:
import pandas as pd
import numpy as np
import re

In [5]:
cno_exp_price = pd.read_csv("datasets/cno_export_prices.csv")
coco_prod = pd.read_csv("datasets/coconut_production.csv")
copra_farmgate = pd.read_csv("datasets/copra_farmgate.csv")
cpo_futures = pd.read_csv("datasets/cpo_futures.csv")
oni_monthly = pd.read_csv("datasets/oni_monthly.csv")
usd_php = pd.read_csv("datasets/usd_php.csv")
others = pd.read_csv("datasets/CMO-Historical-Data-Monthly.csv")

In [7]:
others

,Unnamed: 0,Soybean oil,DAP,Phosphate rock,TSP,Urea,Potassium chloride **,iFATS_OILS
0,2010M01,920.55,383.00,90.00,296.25,266.50,389.38,95.48
1,2010M02,909.19,413.63,90.00,300.00,297.50,374.38,92.84
2,2010M03,911.07,424.50,90.00,354.38,278.70,340.00,91.98
3,2010M04,902.83,417.50,90.00,372.50,253.75,340.00,92.00
4,2010M05,859.49,410.00,90.00,352.75,234.00,334.50,90.81
...,...,...,...,...,...,...,...,...
191,2025M12,1115.67,627.50,152.50,538.50,392.50,358.33,106.37
192,2026M01,1153.54,619.20,152.50,529.20,415.40,366.00,105.82
193,2026M02,1281.96,626.50,152.50,536.25,472.00,372.50,111.18
194,2026M03,1493.09,658.25,152.50,558.13,725.63,380.63,117.76


In [9]:
import pandas as pd
import numpy as np

# 1. Load the data
# The World Bank CSV usually has a BOM character; we strip it and clean headers.
df = pd.read_csv("datasets/CMO-Historical-Data-Monthly.csv")
df.columns = [col.strip().replace('﻿', '') for col in df.columns]
df.rename(columns={df.columns[0]: 'Date'}, inplace=True)

# 2. Define the commodities you need
# You can add more from the CSV headers as needed
commodities = ['Soybean oil', 'DAP', 'Phosphate rock', 'TSP', 'Urea', "Potassium chloride **"]

# 3. Process each into separate DataFrames
# --- SOYBEAN OIL ---
df_soybean_oil = df[['Date', 'Soybean oil']].copy()
df_soybean_oil['Soybean oil'] = pd.to_numeric(df_soybean_oil['Soybean oil'], errors='coerce')
df_soybean_oil['Date'] = pd.to_datetime(df_soybean_oil['Date'], format='%YM%m')
df_soybean_oil.set_index('Date', inplace=True)
df_soybean_oil['MA3'] = df_soybean_oil['Soybean oil'].rolling(window=3).mean()
df_soybean_oil['MA6'] = df_soybean_oil['Soybean oil'].rolling(window=6).mean()
df_soybean_oil['Volatility'] = df_soybean_oil['Soybean oil'].rolling(window=3).std()

# --- DAP (Fertilizer) ---
df_dap = df[['Date', 'DAP']].copy()
df_dap['DAP'] = pd.to_numeric(df_dap['DAP'], errors='coerce')
df_dap['Date'] = pd.to_datetime(df_dap['Date'], format='%YM%m')
df_dap.set_index('Date', inplace=True)
df_dap['MA3'] = df_dap['DAP'].rolling(window=3).mean()
df_dap['MA6'] = df_dap['DAP'].rolling(window=6).mean()

# --- UREA (Fertilizer) ---
df_urea = df[['Date', 'Urea']].copy()
df_urea['Urea'] = pd.to_numeric(df_urea['Urea'], errors='coerce')
df_urea['Date'] = pd.to_datetime(df_urea['Date'], format='%YM%m')
df_urea.set_index('Date', inplace=True)
df_urea['MA3'] = df_urea['Urea'].rolling(window=3).mean()
df_urea['MA6'] = df_urea['Urea'].rolling(window=6).mean()

# --- POTASSIUM CHLORIDE --- 
df_potassium = df[['Date', 'Potassium chloride **']].copy()
df_potassium['Potassium chloride**'] = pd.to_numeric(df_potassium['Potassium chloride **'], errors='coerce')
df_potassium['Date'] = pd.to_datetime(df_potassium['Date'], format='%YM%m')
df_potassium.set_index('Date', inplace=True)
df_potassium['MA3'] = df_potassium['Potassium chloride **'].rolling(window=3).mean()

# Example Output
print("Latest Urea Data:")
print(df_urea.tail())

Latest Urea Data:
              Urea         MA3         MA6
Date                                      
2025-12-01  392.50  398.716667  443.496667
2026-01-01  415.40  405.716667  430.063333
2026-02-01  472.00  426.633333  424.113333
2026-03-01  725.63  537.676667  468.196667
2026-04-01  856.88  684.836667  545.276667


In [ ]:
cno_exp_price["Date"] = pd.to_datetime(cno_exp_price["Date"], errors="coerce")
cno_exp_price["Price"] = pd.to_numeric(cno_exp_price["Price"], errors="coerce")

cno_exp_price = (
    cno_exp_price[["Date", "Price"]]
    .rename(columns={
        "Date": "date",
        "Price": "cno_export_prices"
    })
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

cno_exp_price

In [4]:
coco_prod = pd.read_csv("datasets/coconut_production.csv")
coco_prod = coco_prod[['Year', 'Quarter', 'Volume']]

coco_prod["Year"] = coco_prod["Year"].ffill().astype(int)

coco_prod['Year'] = pd.to_datetime(coco_prod['Year'].astype(str), format='%Y')

quarter_map = {
    'Quarter1': '01',  # Jan
    'Quarter2': '04',  # Apr
    'Quarter3': '07',  # Jul
    'Quarter4': '10'   # Oct
}

coco_prod['Year'] = pd.to_datetime( # Estimates the date as the first day of the quarter
    coco_prod['Year'].dt.year.astype(str) + '-' +
    coco_prod['Quarter'].map(quarter_map) + '-01'
)

coco_prod["Volume"] = pd.to_numeric(coco_prod["Volume"], errors="coerce")

coco_prod = coco_prod.rename(columns={"Volume": "Coconut Production Volume", "Year": "Date"})

coco_prod['Date']
coco_prod


,Date,Quarter,Coconut Production Volume
0,2010-01-01,Quarter1,3571514.43
1,2010-04-01,Quarter2,3564582.99
2,2010-07-01,Quarter3,4032257.30
3,2010-10-01,Quarter4,4024607.15
4,2011-01-01,Quarter1,3350281.44
...,...,...,...
59,2024-10-01,Quarter4,3905697.16
60,2025-01-01,Quarter1,3045870.32
61,2025-04-01,Quarter2,3265637.92
62,2025-07-01,Quarter3,3852636.86


In [5]:
import pandas as pd

copra_farmgate = pd.read_csv("datasets/copra_farmgate.csv")

copra_farmgate = copra_farmgate.melt(
    var_name="date_raw",
    value_name="copra_farmgate"
)

copra_farmgate["date"] = (
    copra_farmgate["date_raw"]
    .str.replace(" Copra Corriente", "", regex=False)
    .str.strip()
)

copra_farmgate["date"] = pd.to_datetime(
    copra_farmgate["date"],
    format="%Y %B"
)

copra_farmgate["copra_farmgate"] = pd.to_numeric(
    copra_farmgate["copra_farmgate"],
    errors="coerce"
)

copra_farmgate = (
    copra_farmgate[["date", "copra_farmgate"]]
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

copra_farmgate

,date,copra_farmgate
0,2010-01-01,13.59
1,2010-02-01,14.47
2,2010-03-01,16.24
3,2010-04-01,17.14
4,2010-05-01,17.65
...,...,...
187,2025-08-01,53.34
188,2025-09-01,51.60
189,2025-10-01,55.83
190,2025-11-01,56.23


In [6]:
import pandas as pd

cpo_futures = pd.read_csv("datasets/cpo_futures.csv")

cpo_futures = cpo_futures[["Date", "Price"]]

cpo_futures["Date"] = pd.to_datetime(cpo_futures["Date"], errors="coerce")

cpo_futures["Price"] = (
    cpo_futures["Price"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

cpo_futures["Price"] = pd.to_numeric(cpo_futures["Price"], errors="coerce")

cpo_futures = (
    cpo_futures
    .rename(columns={
        "Date": "date",
        "Price": "cpo_futures"
    })
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)
cpo_futures

,date,cpo_futures
0,2010-05-30,731.50
1,2010-06-06,723.00
2,2010-06-13,737.25
3,2010-06-20,736.50
4,2010-06-27,723.00
...,...,...
743,2026-03-22,1134.50
744,2026-03-29,1192.00
745,2026-04-05,1156.25
746,2026-04-12,1145.25


In [7]:
import pandas as pd

oni_monthly = pd.read_csv("datasets/oni_monthly.csv")

oni_monthly["date"] = pd.to_datetime(oni_monthly["date"], errors="coerce")
oni_monthly["oni"] = pd.to_numeric(oni_monthly["oni"], errors="coerce")

oni_monthly = (
    oni_monthly[["date", "oni"]]
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

oni_monthly

,date,oni
0,2000-01-01,-1.66
1,2000-02-01,-1.41
2,2000-03-01,-1.07
3,2000-04-01,-0.81
4,2000-05-01,-0.71
...,...,...
309,2025-10-01,-0.51
310,2025-11-01,-0.55
311,2025-12-01,-0.54
312,2026-01-01,-0.37


In [8]:
import pandas as pd

usd_php = pd.read_csv("datasets/usd_php.csv")

usd_php = usd_php[["Date", "Price"]]

usd_php["Date"] = pd.to_datetime(usd_php["Date"], errors="coerce")
usd_php["Price"] = pd.to_numeric(usd_php["Price"], errors="coerce")

usd_php = (
    usd_php
    .rename(columns={
        "Date": "date",
        "Price": "usd_php"
    })
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

In [9]:
import pandas as pd

df = pd.DataFrame()

df["date"] = pd.date_range(
    start="2010-01-01",
    end="2024-12-30",
    freq="W"   # weekly
)


df = df.set_index("date").sort_index()
df_monthly = df.asfreq("MS")  # MS = month start
print(df.head())
print(df.tail())

Empty DataFrame
Columns: []
Index: [2010-01-03 00:00:00, 2010-01-10 00:00:00, 2010-01-17 00:00:00, 2010-01-24 00:00:00, 2010-01-31 00:00:00]
Empty DataFrame
Columns: []
Index: [2024-12-01 00:00:00, 2024-12-08 00:00:00, 2024-12-15 00:00:00, 2024-12-22 00:00:00, 2024-12-29 00:00:00]


In [10]:
import pandas as pd
import numpy as np
import re

# ─────────────────────────────────────────────
# 1. LOAD RAW DATASETS
# ─────────────────────────────────────────────

cno_export_prices = pd.read_csv("datasets/cno_export_prices.csv")
coco_prod         = pd.read_csv("datasets/coconut_production.csv")
copra_farmgate    = pd.read_csv("datasets/copra_farmgate.csv")
cpo_futures       = pd.read_csv("datasets/cpo_futures.csv")
oni_monthly       = pd.read_csv("datasets/oni_monthly.csv")
usd_php           = pd.read_csv("datasets/usd_php.csv")

# ─────────────────────────────────────────────
# 2. CLEAN: CNO EXPORT PRICES
# ─────────────────────────────────────────────

cno_export_prices["Date"]  = pd.to_datetime(cno_export_prices["Date"], errors="coerce")
cno_export_prices["Price"] = pd.to_numeric(cno_export_prices["Price"], errors="coerce")

cno_export_prices = (
    cno_export_prices[["Date", "Price"]]
    .rename(columns={"Date": "date", "Price": "cno_export_prices"})
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print("CNO Export Prices:")
print(cno_export_prices.head(3))

# ─────────────────────────────────────────────
# 3. CLEAN: COCONUT PRODUCTION (QUARTERLY)
# ─────────────────────────────────────────────

coco_prod = coco_prod[["Year", "Quarter", "Volume"]]
coco_prod["Year"] = coco_prod["Year"].ffill().astype(int)

quarter_map = {
    "Quarter1": "01",
    "Quarter2": "04",
    "Quarter3": "07",
    "Quarter4": "10"
}

coco_prod["date"] = pd.to_datetime(
    coco_prod["Year"].astype(str) + "-" +
    coco_prod["Quarter"].map(quarter_map) + "-01"
)

coco_prod["Volume"] = pd.to_numeric(coco_prod["Volume"], errors="coerce")

coco_prod = (
    coco_prod
    .rename(columns={"Volume": "coconut_production"})
    .drop(columns=["Year", "Quarter"])
    [["date", "coconut_production"]]
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nCoconut Production:")
print(coco_prod.head(3))

# ─────────────────────────────────────────────
# 4. CLEAN: COPRA FARMGATE (MONTHLY)
# ─────────────────────────────────────────────

copra_farmgate = copra_farmgate.melt(
    var_name="date_raw",
    value_name="copra_farmgate"
)

copra_farmgate["date"] = (
    copra_farmgate["date_raw"]
    .str.replace(" Copra Corriente", "", regex=False)
    .str.strip()
)

copra_farmgate["date"] = pd.to_datetime(
    copra_farmgate["date"],
    format="%Y %B",
    errors="coerce"
)

copra_farmgate["copra_farmgate"] = pd.to_numeric(
    copra_farmgate["copra_farmgate"], errors="coerce"
)

copra_farmgate = (
    copra_farmgate[["date", "copra_farmgate"]]
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nCopra Farmgate:")
print(copra_farmgate.head(3))

# ─────────────────────────────────────────────
# 5. CLEAN: CPO FUTURES (WEEKLY)
# ─────────────────────────────────────────────

cpo_futures = cpo_futures[["Date", "Price"]]

cpo_futures["Date"] = pd.to_datetime(cpo_futures["Date"], errors="coerce")

cpo_futures["Price"] = (
    cpo_futures["Price"]
    .astype(str)
    .str.replace(",", "", regex=False)
)
cpo_futures["Price"] = pd.to_numeric(cpo_futures["Price"], errors="coerce")

cpo_futures = (
    cpo_futures
    .rename(columns={"Date": "date", "Price": "cpo_futures"})
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nCPO Futures:")
print(cpo_futures.head(3))

# ─────────────────────────────────────────────
# 6. CLEAN: ONI MONTHLY
# ─────────────────────────────────────────────

oni_monthly["date"] = pd.to_datetime(oni_monthly["date"], errors="coerce")
oni_monthly["oni"]  = pd.to_numeric(oni_monthly["oni"], errors="coerce")

oni_monthly = (
    oni_monthly[["date", "oni"]]
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nONI Monthly:")
print(oni_monthly.head(3))

# ─────────────────────────────────────────────
# 7. CLEAN: USD/PHP (WEEKLY)
# ─────────────────────────────────────────────

usd_php = usd_php[["Date", "Price"]]

usd_php["Date"]  = pd.to_datetime(usd_php["Date"], errors="coerce")
usd_php["Price"] = pd.to_numeric(usd_php["Price"], errors="coerce")

usd_php = (
    usd_php
    .rename(columns={"Date": "date", "Price": "usd_php"})
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nUSD/PHP:")
print(usd_php.head(3))

# ─────────────────────────────────────────────
# 8. BUILD WEEKLY SPINE
# ─────────────────────────────────────────────

weekly_spine = pd.DataFrame({
    "date": pd.date_range(start="2010-06-01", end="2024-12-31", freq="W-MON")
})

print(f"\nWeekly spine: {len(weekly_spine)} rows")
print(weekly_spine.head(3))

# ─────────────────────────────────────────────
# 9. MERGE ALL SERIES VIA merge_asof
# Handles both day-of-week mismatch (weekly series)
# and monthly/quarterly frequency differences
# direction="backward" = forward-fill by design
# ─────────────────────────────────────────────

df = weekly_spine.copy().sort_values("date")

# Weekly series
df = pd.merge_asof(df, cpo_futures.sort_values("date"),
                   on="date", direction="backward")

df = pd.merge_asof(df, usd_php.sort_values("date"),
                   on="date", direction="backward")

# Monthly series
df = pd.merge_asof(df, cno_export_prices.sort_values("date"),
                   on="date", direction="backward")

df = pd.merge_asof(df, copra_farmgate.sort_values("date"),
                   on="date", direction="backward")

df = pd.merge_asof(df, oni_monthly.sort_values("date"),
                   on="date", direction="backward")

# Quarterly series
df = pd.merge_asof(df, coco_prod.sort_values("date"),
                   on="date", direction="backward")

# ─────────────────────────────────────────────
# 10. DROP ROWS BEFORE 2010 AND AFTER 2024
# ─────────────────────────────────────────────

df = df[
    (df["date"] >= "2010-01-01") &
    (df["date"] <= "2024-12-31")
].reset_index(drop=True)

# ─────────────────────────────────────────────
# 11. DROP ROWS WHERE CRITICAL SERIES ARE MISSING
# CNO and copra are non-negotiable
# ─────────────────────────────────────────────

print("\nMissing values before dropping critical rows:")
print(df.isnull().sum())

df.dropna(subset=["cno_export_prices", "copra_farmgate"], inplace=True)
df = df.reset_index(drop=True)

# ─────────────────────────────────────────────
# 12. SANITY CHECKS
# ─────────────────────────────────────────────

print("\n" + "=" * 50)
print("SANITY CHECKS")
print("=" * 50)

print(f"Final shape:  {df.shape}")
print(f"Date range:   {df['date'].min().date()} to {df['date'].max().date()}")

assert df["date"].is_monotonic_increasing, "ERROR: dates not sorted"
print("Date order:   OK")

dupes = df["date"].duplicated().sum()
assert dupes == 0, f"ERROR: {dupes} duplicate dates found"
print("Duplicates:   OK")

ratio = df["cno_export_prices"] / df["copra_farmgate"]
print(f"\nCNO/Copra ratio:")
print(f"  Mean: {ratio.mean():.2f} | Min: {ratio.min():.2f} | Max: {ratio.max():.2f}")

if not (1.5 <= ratio.mean() <= 4.0):
    print("  WARNING: ratio outside expected range — check units")
else:
    print("  Units look correct")

print(f"\nMissing values after cleaning:")
print(df.isnull().sum())

print(f"\nFinal dataset preview:")
print(df.head(8).to_string())

# ─────────────────────────────────────────────
# 13. SAVE
# ─────────────────────────────────────────────

df.to_csv("datasets/merged_weekly.csv", index=False)
print(f"\nSaved to datasets/merged_weekly.csv")

# Convert copra from PHP/kg to USD/MT for ratio check and SCFM
df["copra_usd_mt"] = (df["copra_farmgate"] * 1000) / df["usd_php"]

# Verify ratio now
ratio = df["cno_export_prices"] / df["copra_usd_mt"]
print(f"CNO/Copra ratio (USD/MT basis):")
print(f"  Mean: {ratio.mean():.2f} | Min: {ratio.min():.2f} | Max: {ratio.max():.2f}")


CNO Export Prices:
        date  cno_export_prices
0 2000-01-01              654.0
1 2000-02-02              591.0
2 2000-03-05              552.0

Coconut Production:
        date  coconut_production
0 2010-01-01          3571514.43
1 2010-04-01          3564582.99
2 2010-07-01          4032257.30

Copra Farmgate:
        date  copra_farmgate
0 2010-01-01           13.59
1 2010-02-01           14.47
2 2010-03-01           16.24

CPO Futures:
        date  cpo_futures
0 2010-05-30       731.50
1 2010-06-06       723.00
2 2010-06-13       737.25

ONI Monthly:
        date   oni
0 2000-01-01 -1.66
1 2000-02-01 -1.41
2 2000-03-01 -1.07

USD/PHP:
        date  usd_php
0 2000-01-30    40.60
1 2000-02-06    40.48
2 2000-02-13    40.70

Weekly spine: 761 rows
        date
0 2010-06-07
1 2010-06-14
2 2010-06-21

Missing values before dropping critical rows:
date                  0
cpo_futures           0
usd_php               0
cno_export_prices     0
copra_farmgate        0
oni               

C:\Users\Tyrese\AppData\Local\Temp\ipykernel_43436\2180914809.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cno_export_prices["Date"]  = pd.to_datetime(cno_export_prices["Date"], errors="coerce")


In [11]:
print("CPO date range:", cpo_futures["date"].min(), "to", cpo_futures["date"].max())
print("CPO sample dates:", cpo_futures["date"].head(5).tolist())
print()
print("USD/PHP date range:", usd_php["date"].min(), "to", usd_php["date"].max())
print("USD/PHP sample dates:", usd_php["date"].head(5).tolist())
print()
print("Weekly spine sample:", weekly_spine["date"].head(5).tolist())

CPO date range: 2010-05-30 00:00:00 to 2026-04-19 00:00:00
CPO sample dates: [Timestamp('2010-05-30 00:00:00'), Timestamp('2010-06-06 00:00:00'), Timestamp('2010-06-13 00:00:00'), Timestamp('2010-06-20 00:00:00'), Timestamp('2010-06-27 00:00:00')]

USD/PHP date range: 2000-01-30 00:00:00 to 2026-04-19 00:00:00
USD/PHP sample dates: [Timestamp('2000-01-30 00:00:00'), Timestamp('2000-02-06 00:00:00'), Timestamp('2000-02-13 00:00:00'), Timestamp('2000-02-20 00:00:00'), Timestamp('2000-02-27 00:00:00')]

Weekly spine sample: [Timestamp('2010-06-07 00:00:00'), Timestamp('2010-06-14 00:00:00'), Timestamp('2010-06-21 00:00:00'), Timestamp('2010-06-28 00:00:00'), Timestamp('2010-07-05 00:00:00')]


In [12]:
df["scfm"] = df["cno_export_prices"] - 6.3 * df["copra_usd_mt"]  # Supply Chain Friction Margin

# Lag features — what did the market look like 1, 2, 4 weeks ago?
df["scfm_lag1"] = df["scfm"].shift(1)
df["scfm_lag2"] = df["scfm"].shift(2)
df["scfm_lag4"] = df["scfm"].shift(4)

# Rolling features — what is the trend and volatility?
df["scfm_mean12"] = df["scfm"].shift(1).rolling(12).mean()
df["scfm_std12"]  = df["scfm"].shift(1).rolling(12).std()

# CPO returns
df["cpo_ret1w"] = df["cpo_futures"].pct_change(1).shift(1)
df["cpo_ret4w"] = df["cpo_futures"].pct_change(4).shift(1)

# ENSO phase
def oni_phase(val):
    if val >=  0.5: return  1   # El Nino
    if val <= -0.5: return -1   # La Nina
    return 0                    # Neutral

df["oni_lag13"] = df["oni"].shift(13)   # 3-month crop lag
df["oni_phase"] = df["oni_lag13"].apply(oni_phase)

# Seasonality
df["month"] = df["date"].dt.month

# Target variable — what you are trying to predict
# 4-week forward return of CNO export price
df["target"] = df["cno_export_prices"].pct_change(4).shift(-4)

In [13]:
df.dropna(inplace=True)
df = df.reset_index(drop=True)

In [14]:
feature_cols = [
    "scfm", "scfm_lag1", "scfm_lag2", "scfm_lag4",
    "scfm_mean12", "scfm_std12",
    "cpo_ret1w", "cpo_ret4w",
    "oni_lag13", "oni_phase",
    "usd_php", "coconut_production",
    "month"
]

X = df[feature_cols]
y = df["target"]

In [15]:
cutoff = "2022-01-01"   # last 3 years as test

X_train = X[df["date"] < cutoff]
X_test  = X[df["date"] >= cutoff]
y_train = y[df["date"] < cutoff]
y_test  = y[df["date"] >= cutoff]

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

Train: 591 rows | Test: 153 rows


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit ONLY on train
X_test_scaled  = scaler.transform(X_test)        # transform only

In [18]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Baseline: KNN
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Primary: XGBoost
xgb = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
xgb.fit(X_train, y_train)   # no scaling needed

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [19]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def evaluate(name, model, X_t, y_t):
    preds = model.predict(X_t)
    rmse  = np.sqrt(mean_squared_error(y_t, preds))
    mae   = mean_absolute_error(y_t, preds)
    da    = np.mean(np.sign(preds) == np.sign(y_t))   # directional accuracy
    print(f"{name:10} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | Direction: {da:.2%}")

evaluate("KNN",     knn, X_test_scaled, y_test)
evaluate("XGBoost", xgb, X_test,        y_test)

KNN        | RMSE: 0.0893 | MAE: 0.0669 | Direction: 48.37%
XGBoost    | RMSE: 0.0980 | MAE: 0.0755 | Direction: 39.87%


# FIXED ROLLING STUFF

In [26]:
# Price lags — capture autocorrelation in price series
for col in ['cpo_futures',  'cno_export_prices',  'copra_farmgate']:
    for lag in [1, 2, 4, 8, 13, 26, 52]:  # weeks
        df[f"{col}_lag{lag}w"] = df[col].shift(lag)

# ONI lags — for how the climate does stuff to the out
# this is for ENSO sensitivity analysis
for lag in [8, 13, 18, 26, 52]:
    df[f"ONI_lag{lag}w"] = df["oni"].shift(lag)

# Production lag 
for lag in [13, 26, 39, 52]:
    df[f"Production_lag{lag}w"] = df["coconut_production"].shift(lag)

In [27]:
new_cols = {}

for col in ['cpo_futures',  'cno_export_prices',  'copra_farmgate', 'oni']:
    for window in [4, 8, 13, 26, 52]:
        new_cols[f"{col}_roll_mean_{window}w"] = df[col].rolling(window).mean()
        new_cols[f"{col}_roll_std_{window}w"]  = df[col].rolling(window).std()
        new_cols[f"{col}_roll_min_{window}w"]  = df[col].rolling(window).min()
        new_cols[f"{col}_roll_max_{window}w"]  = df[col].rolling(window).max()

# Add everything at once
df = pd.concat([df, pd.DataFrame(new_cols)], axis=1)

In [28]:
df

,date,cpo_futures,usd_php,cno_export_prices,copra_farmgate,oni,coconut_production,copra_usd_mt,scfm,scfm_lag1,...,oni_roll_min_13w,oni_roll_max_13w,oni_roll_mean_26w,oni_roll_std_26w,oni_roll_min_26w,oni_roll_max_26w,oni_roll_mean_52w,oni_roll_std_52w,oni_roll_min_52w,oni_roll_max_52w
0,2010-09-06,846.75,43.905,798.00,24.83,-1.56,4032257.30,565.539232,-2764.897164,-2428.846543,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-09-13,847.50,44.215,921.00,24.83,-1.56,4032257.30,561.574126,-2616.916996,-2764.897164,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-09-20,889.25,43.805,921.00,24.83,-1.56,4032257.30,566.830271,-2650.030704,-2616.916996,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-09-27,856.00,43.745,921.00,24.83,-1.56,4032257.30,567.607727,-2654.928678,-2650.030704,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-10-04,890.50,43.270,921.00,27.62,-1.64,4024607.15,638.317541,-3100.400508,-2654.928678,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
739,2024-11-04,1145.00,58.460,1099.09,51.68,-0.30,3905697.16,884.023264,-4470.256562,-2836.456775,...,-0.30,-0.07,0.018462,0.233884,-0.30,0.49,0.776923,0.860067,-0.30,2.06
740,2024-11-11,1128.00,58.730,1099.09,51.68,-0.30,3905697.16,879.959135,-4444.652551,-4470.256562,...,-0.30,-0.07,-0.011923,0.221143,-0.30,0.49,0.732885,0.855351,-0.30,2.06
741,2024-11-18,1094.75,58.960,1071.67,51.68,-0.30,3905697.16,876.526459,-4450.446689,-4444.652551,...,-0.30,-0.07,-0.042308,0.202944,-0.30,0.49,0.688846,0.848280,-0.30,2.06
742,2024-11-25,1106.00,58.620,1071.67,51.68,-0.30,3905697.16,881.610372,-4482.475343,-4450.446689,...,-0.30,-0.17,-0.072692,0.177619,-0.30,0.22,0.644808,0.838796,-0.30,2.06
